# 10 — Final test (scored once)

The three frozen models plus an **equal-weight ensemble** (declared before opening the test) are scored on the untouched test year, Oct 2024 → Sep 2025. Nothing is chosen here.

- **Input:** dataset (05), naive forecasts (06), model configs from 07–09
- **Output:** `data/processed/final_test_forecasts.parquet`, `results/10_final_test_*` (metrics, Diebold-Mariano tests, protocol)

| Model | MAE | RMSE | rMAE |
|---|---|---|---|
| naive_epf | 12.41 | 19.12 | 1.000 |
| linear | 9.11 | 13.02 | 0.734 |
| XGBoost | 8.98 ± 0.03 | 12.97 | 0.724 |
| LEAR | 8.28 | 12.17 | 0.667 |
| **ensemble** | **8.17 ± 0.01** | **11.84** | **0.659** |

- **Conclusion:** XGBoost does not beat the linear model detectably; LEAR beats both (p ≈ 0); the ensemble's edge over LEAR is not significant. LEAR's jump is explained by bias–variance: with 246 coefficients, one more year of data cuts estimation variance.

In [1]:
# --- 10_final_test: the test year, scored once ------------------------------------
# Rule of this notebook: nothing is CHOSEN here. The three models were frozen on the
# validation year (07, 08, 09) and the ensemble was declared on 18/09, before the
# first test forecast existed. This cell only rebuilds the inputs.
%load_ext autoreload
%autoreload 2

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import forecasting as fc

TARGET = fc.TARGET

# --- Dataset, naive forecasts, calendar: exactly as in 07 / 08 / 09 ---------------
dataset = pd.read_parquet(fc.PROCESSED / "dataset_hourly_2021_2025q3.parquet")
naive = pd.read_parquet(fc.PROCESSED / "naive_forecasts_2021_2025q3.parquet")

NAIVE_COLUMNS = ["naive_d1", "naive_d7", "naive_mean7", "naive_epf"]
dataset = dataset.merge(naive[["ts_utc"] + NAIVE_COLUMNS], on="ts_utc",
                        how="left", validate="one_to_one")

dataset = fc.add_split_columns(dataset)
dataset = fc.add_calendar_features(dataset)

# Engineered feature shared by linear_v3 and xgb_single (residual load in GW)
dataset["gas_x_residual_load"] = (
    dataset["gas_price_d1_eur_mwh"] * dataset["residual_load_mw"] / 1000
)

# --- The four declared candidates --------------------------------------------------
# Written down BEFORE any test forecast exists: name -> where it was chosen, protocol.
xgb_config = json.loads((fc.RESULTS / "08_xgboost_config.json").read_text())
lear_config = json.loads((fc.RESULTS / "09_lear_config.json").read_text())

CANDIDATES = {
    "linear_v3_avg": "07 - daily refit, mean of expanding + rolling 365",
    "xgb_single":    f"08 - refit every {xgb_config['refit_every_days']} d, expanding, {len(xgb_config['seeds_checked'])} seeds",
    "lear_v1_avg":   f"09 - refit every {lear_config['refit_every_days']} d, mean of expanding + rolling 365",
    "ensemble_avg":  "declared 18/09 - equal-weighted mean of the three above",
}
assert len(CANDIDATES) == 4, "the candidate set is frozen: do not add a fifth one here"

# Declared, NOT corrected: the linear model refits every day, the other two every 30.
# Measured on validation in 08: linear_v3_avg 9.32 vs linear_v3_avg_refit30 9.42,
# i.e. ~0.10 EUR/MWh of head start IN FAVOUR of the linear model.
REFIT_HANDICAP_EUR_MWH = 0.10

# --- Guard: in this notebook a back-test may only ever run on the test split --------
def walk_forward_test(fit, predict, label, window_days=None, refit_every_days=1):
    """fc.walk_forward locked on split='test', with shape checks.

    Every forecast of this notebook goes through here, so no cell can quietly re-run
    a model on the validation year and retune something on it. The day-by-day leakage
    guard inside fc.walk_forward still applies. On the test year the expanding window
    is train + validation, which is correct: on 1 Oct 2024 a real desk did have every
    price up to 30 Sep 2024.
    """
    start = time.perf_counter()
    forecast = fc.walk_forward(dataset, fit, predict, split="test",
                               window_days=window_days,
                               refit_every_days=refit_every_days)
    is_test = dataset["split"] == "test"
    assert forecast[is_test].notna().all(), "some test hours were not forecast"
    assert forecast[~is_test].isna().all(), "a forecast leaked outside the test split"
    print(f"{label}: {time.perf_counter() - start:.0f} s")
    return forecast


# --- Checks ------------------------------------------------------------------------
assert len(dataset) == 41615
assert dataset.loc[dataset["split"] != "train", NAIVE_COLUMNS].notna().all().all()

test = dataset.loc[dataset["split"] == "test"]
assert len(test) == 8760

first_test_day = test["local_date"].min()
print(dataset["split"].value_counts().to_string())
print(f"\ntest: {first_test_day:%Y-%m-%d} -> {test['local_date'].max():%Y-%m-%d}, "
      f"{test['local_date'].nunique()} days")
print(f"training hours available on the first test day: "
      f"{(dataset['local_date'] < first_test_day).sum()}")
print("\ndeclared candidates:")
for name, description in CANDIDATES.items():
    print(f"  {name:<15} {description}")

split
train         24071
validation     8784
test           8760

test: 2024-10-01 -> 2025-09-30, 365 days
training hours available on the first test day: 32855

declared candidates:
  linear_v3_avg   07 - daily refit, mean of expanding + rolling 365
  xgb_single      08 - refit every 30 d, expanding, 5 seeds
  lear_v1_avg     09 - refit every 30 d, mean of expanding + rolling 365
  ensemble_avg    declared 18/09 - equal-weighted mean of the three above


In [2]:
# --- The ensemble on the validation year: a number to read, not a choice -----------
# ensemble_avg = equal-weighted mean of the three chosen models. Zero parameters are
# estimated, so this costs no degree of freedom: it is NOT a sixth selection step.
# Declared on 18/09: the ensemble goes to the test set whatever this cell prints.
SEEDS = xgb_config["seeds_checked"]
xgb_seed_columns = [f"xgb_single_seed{seed}" for seed in SEEDS]

VALIDATION_FILES = {
    "linear_validation_forecasts.parquet": ["linear_v3_avg"],
    "lear_validation_forecasts.parquet": ["lear_v1_avg"],
    "xgb_validation_forecasts.parquet": ["xgb_single"] + xgb_seed_columns,
}
for filename, columns in VALIDATION_FILES.items():
    saved = pd.read_parquet(fc.PROCESSED / filename)
    dataset = dataset.merge(saved[["ts_utc"] + columns], on="ts_utc",
                            how="left", validate="one_to_one")

# These files hold validation rows only: nothing must appear outside that split
COMPONENTS = ["linear_v3_avg", "xgb_single", "lear_v1_avg"]
is_validation = dataset["split"] == "validation"
assert dataset.loc[is_validation, COMPONENTS].notna().all().all()
assert dataset.loc[~is_validation, COMPONENTS].isna().all().all()

# One ensemble PER SEED: averaging a lucky seed into the ensemble and then comparing
# it with a seed-averaged XGBoost would tilt the comparison towards the ensemble.
ensemble_columns = []
for seed in SEEDS:
    column = f"ensemble_avg_seed{seed}"
    dataset[column] = dataset[["linear_v3_avg", f"xgb_single_seed{seed}",
                               "lear_v1_avg"]].mean(axis=1)
    ensemble_columns.append(column)

# --- Metrics ------------------------------------------------------------------------
metrics = fc.evaluate(
    dataset,
    ["naive_epf", "linear_v3_avg", "lear_v1_avg"] + xgb_seed_columns + ensemble_columns,
    split="validation",
)
overall = metrics.loc[metrics["period"] == "all"].set_index("forecast")[["MAE", "RMSE", "rMAE"]]

print(overall.loc[["naive_epf", "linear_v3_avg", "lear_v1_avg"]].round(3).to_string())
print("\nXGBoost, mean +- std across the 5 seeds:")
print(overall.loc[xgb_seed_columns].agg(["mean", "std"]).round(3).to_string())
print("\nensemble, mean +- std across the 5 seeds:")
print(overall.loc[ensemble_columns].agg(["mean", "std"]).round(3).to_string())

# --- Does the ensemble beat its best single component, for EVERY seed? --------------
print()
print(pd.DataFrame([
    fc.diebold_mariano(dataset, "lear_v1_avg", column, split="validation", loss=loss)
    for column in ensemble_columns
    for loss in ["abs", "squared"]
]).drop(columns=["forecast_a", "n_days"]).round(4).to_string(index=False))

                  MAE    RMSE   rMAE
forecast                            
naive_epf      12.296  18.352  1.000
linear_v3_avg   9.317  13.084  0.758
lear_v1_avg     9.030  12.891  0.734

XGBoost, mean +- std across the 5 seeds:
        MAE    RMSE   rMAE
mean  9.127  12.956  0.742
std   0.016   0.016  0.001

ensemble, mean +- std across the 5 seeds:
        MAE    RMSE   rMAE
mean  8.373  11.837  0.681
std   0.005   0.005  0.000

        forecast_b    loss  mean_daily_diff  dm_stat  p_value_b_better
ensemble_avg_seed0     abs           0.6591   4.9679            0.0000
ensemble_avg_seed0 squared          26.1129   3.0521            0.0011
ensemble_avg_seed1     abs           0.6559   4.9047            0.0000
ensemble_avg_seed1 squared          26.0220   3.0259            0.0012
ensemble_avg_seed2     abs           0.6640   4.9891            0.0000
ensemble_avg_seed2 squared          26.1494   3.0615            0.0011
ensemble_avg_seed3     abs           0.6532   4.9097            0.0000

In [3]:
# --- Why does the ensemble work? The arithmetic of error correlation ----------------
# Forecast correlation is useless here: any two decent forecasts of the same series
# correlate ~0.99. What decides whether averaging helps is how much their ERRORS have
# in common. This is portfolio variance with equal weights: the errors are the assets.
validation = dataset.loc[dataset["split"] == "validation"]
COMPONENTS_SEED0 = ["linear_v3_avg", "xgb_single_seed0", "lear_v1_avg"]

errors = validation[COMPONENTS_SEED0].apply(lambda column: validation[TARGET] - column)

print("forecast correlation (uninformative, everything sits at ~0.99):")
print(validation[COMPONENTS_SEED0].corr().round(3).to_string())
print("\nERROR correlation (this is the one that matters):")
print(errors.corr().round(3).to_string())

# --- Bias or variance? Averaging diversifies variance, never a shared bias ----------
# A bias common to all three is correlated 1.00 by construction, so the mean keeps it.
# Only the part where the models DISAGREE can be diversified away.
print("\nmean error (bias) and error std, EUR/MWh:")
print(pd.DataFrame({"bias": errors.mean(), "std": errors.std(ddof=0)}).round(3).to_string())

# --- Exact decomposition: a pipeline check, not a prediction -------------------------
# The ensemble error IS the mean of the component errors, so
#     RMSE(ensemble)^2 = (1 / n^2) * sum_ij E[e_i e_j]
# holds as an identity. Using raw second moments rather than covariances keeps the
# bias inside, so the number is directly comparable with the RMSE. A mismatch here
# would mean the ensemble column was not built from these three columns.
E = errors.to_numpy()
second_moments = E.T @ E / len(E)                 # E[e_i e_j], bias included
rmse_from_matrix = np.sqrt(second_moments.mean())  # equal weights = mean of the matrix
rmse_observed = np.sqrt(((validation[TARGET] - validation["ensemble_avg_seed0"]) ** 2).mean())
print(f"\nRMSE from the second-moment matrix {rmse_from_matrix:.4f} "
      f"vs observed {rmse_observed:.4f}")
assert abs(rmse_from_matrix - rmse_observed) < 1e-9, "the ensemble is not the mean of these three"

# --- How much diversification is still on the table? --------------------------------
# With n models of similar accuracy and average pairwise ERROR correlation rho:
#     sd(ensemble) / sd(single) = sqrt((1 + (n - 1) * rho) / n)
# As n grows this ratio tends to sqrt(rho): correlation is a FLOOR no number of extra
# models can break. A fourth model similar to those already in (rho ~ 0.87) would buy
# almost nothing; a decorrelated one is what pays. This is the design tool.
n_models = len(COMPONENTS_SEED0)
rho_bar = errors.corr().to_numpy()[np.triu_indices(n_models, k=1)].mean()
ratio_now = np.sqrt((1 + (n_models - 1) * rho_bar) / n_models)

print(f"\naverage pairwise error correlation : {rho_bar:.3f}")
print(f"sd ratio with {n_models} models          : {ratio_now:.3f}  "
      f"({100 * (ratio_now - 1):+.1f}%)")
print(f"floor with infinitely many models  : {np.sqrt(rho_bar):.3f}  "
      f"({100 * (np.sqrt(rho_bar) - 1):+.1f}%)")
print(f"share of the available gain already captured: "
      f"{(1 - ratio_now) / (1 - np.sqrt(rho_bar)):.0%}")

forecast correlation (uninformative, everything sits at ~0.99):
                  linear_v3_avg  xgb_single_seed0  lear_v1_avg
linear_v3_avg             1.000             0.971        0.935
xgb_single_seed0          0.971             1.000        0.938
lear_v1_avg               0.935             0.938        1.000

ERROR correlation (this is the one that matters):
                  linear_v3_avg  xgb_single_seed0  lear_v1_avg
linear_v3_avg             1.000             0.879        0.673
xgb_single_seed0          0.879             1.000        0.692
lear_v1_avg               0.673             0.692        1.000

mean error (bias) and error std, EUR/MWh:
                   bias     std
linear_v3_avg     0.126  13.083
xgb_single_seed0  0.201  12.940
lear_v1_avg      -0.095  12.890

RMSE from the second-moment matrix 11.8339 vs observed 11.8339

average pairwise error correlation : 0.748
sd ratio with 3 models          : 0.912  (-8.8%)
floor with infinitely many models  : 0.865  (-13.5%)


In [4]:
# --- Test forecasts, part 1: linear_v3_avg and xgb_single ---------------------------
# Nothing is scored in this cell: all four candidates are generated first, then the
# test split is evaluated exactly once. Column names are prefixed "test_" because the
# validation forecasts merged earlier already occupy the plain names.
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor

FEATURES_V3 = [
    "naive_d1",
    "naive_d7",
    "residual_load_mw",
    "gas_price_d1_eur_mwh",
    "is_holiday",
    "gas_x_residual_load",
    "yesterday_holiday",
]
ROLLING_WINDOW_DAYS = 365


def make_hourly_linear(features: list):
    """24 hourly linear regressions: same factory as 07 and 08, without asinh."""

    def design(df: pd.DataFrame) -> pd.DataFrame:
        X = df[features].astype(float)
        # Fixed day-of-week columns (Tuesday..Sunday), Monday = reference category
        for day in range(1, 7):
            X[f"dow_{day}"] = (df["day_of_week"] == day).astype(float)
        return X

    def fit(train: pd.DataFrame) -> dict:
        models = {}
        for hour, rows in train.groupby("local_hour"):
            rows = rows.dropna(subset=features)   # first week of 2021: no naive_d7
            models[hour] = LinearRegression().fit(design(rows), rows[TARGET])
        return models

    def predict(models: dict, day: pd.DataFrame) -> np.ndarray:
        forecast = pd.Series(np.nan, index=day.index)
        for hour, rows in day.groupby("local_hour"):
            forecast.loc[rows.index] = models[hour].predict(design(rows))
        return forecast.to_numpy()

    return fit, predict


def make_xgb(features: list, params: dict, per_hour: bool):
    """Same factory as 08: 24 hourly models, or one model that splits on local_hour."""

    def fit(train: pd.DataFrame) -> dict:
        train = train.dropna(subset=features)
        if not per_hour:
            return {"all": XGBRegressor(**params).fit(
                train[features].astype(float), train[TARGET])}
        return {hour: XGBRegressor(**params).fit(rows[features].astype(float), rows[TARGET])
                for hour, rows in train.groupby("local_hour")}

    def predict(models: dict, day: pd.DataFrame) -> np.ndarray:
        if not per_hour:
            return models["all"].predict(day[features].astype(float))
        forecast = pd.Series(np.nan, index=day.index)
        for hour, rows in day.groupby("local_hour"):
            forecast.loc[rows.index] = models[hour].predict(rows[features].astype(float))
        return forecast.to_numpy()

    return fit, predict


# --- linear_v3_avg: daily refit, mean of expanding and rolling 365 ------------------
fit_linear, predict_linear = make_hourly_linear(FEATURES_V3)

dataset["test_linear_expanding"] = walk_forward_test(
    fit_linear, predict_linear, label="linear expanding",
)
dataset["test_linear_roll365"] = walk_forward_test(
    fit_linear, predict_linear, label="linear rolling 365",
    window_days=ROLLING_WINDOW_DAYS,
)
dataset["test_linear_v3_avg"] = dataset[
    ["test_linear_expanding", "test_linear_roll365"]].mean(axis=1)

# --- xgb_single: expanding, refit every 30 days, one walk-forward per seed ----------
# The frozen config must match the feature list used above, or the two models would
# no longer share the same information set
assert xgb_config["features"] == FEATURES_V3 + ["day_of_week", "local_hour"]
assert xgb_config["window_days"] is None, "xgb_single is the expanding-window version"

test_xgb_columns = []
for seed in SEEDS:
    params = {**xgb_config["params"], "random_state": seed}
    fit_xgb, predict_xgb = make_xgb(xgb_config["features"], params, xgb_config["per_hour"])
    column = f"test_xgb_single_seed{seed}"
    dataset[column] = walk_forward_test(
        fit_xgb, predict_xgb, label=f"xgb seed {seed}",
        refit_every_days=xgb_config["refit_every_days"],
    )
    test_xgb_columns.append(column)

print("\nlinear and XGBoost test forecasts generated, not scored yet")

linear expanding: 32 s
linear rolling 365: 28 s
xgb seed 0: 4 s
xgb seed 1: 4 s
xgb seed 2: 4 s
xgb seed 3: 4 s
xgb seed 4: 4 s

linear and XGBoost test forecasts generated, not scored yet


In [5]:
# --- Test forecasts, part 2: lear_v1_avg, then the ensembles ------------------------
# The LEAR does not share the five features of the other two: it needs full daily
# profiles of prices and fundamentals. Rebuilt exactly as in 09 -- same pivot, same
# DST convention as the naive forecasts of 06.
from sklearn.linear_model import LassoLarsIC


def daily_profile(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """One row per local day, one column per local hour.

    25-hour days average the two 02:00 rows, 23-hour days interpolate the missing
    one, so p_d1_h19 is exactly the number naive_d1 uses at 19:00.
    """
    profile = df.pivot_table(index="local_date", columns="local_hour",
                             values=column, aggfunc="mean")
    profile = profile.T.interpolate().T
    assert profile.shape == (1734, 24), profile.shape
    assert profile.notna().all().all(), f"holes in the daily profile of {column}"
    # shift(k) means "k days earlier" only if no calendar day is missing
    day_steps = profile.index.to_series().diff().dropna()
    assert (day_steps == pd.Timedelta(days=1)).all(), "a calendar day is missing"
    return profile


def lagged_block(profile: pd.DataFrame, lags: list, prefix: str) -> pd.DataFrame:
    """Flatten `profile` shifted by each lag into columns <prefix>_d<lag>_h<hour>."""
    columns = {}
    for lag in lags:
        shifted = profile.shift(lag)
        for hour in range(24):
            columns[f"{prefix}_d{lag}_h{hour:02d}"] = shifted[hour]
    return pd.DataFrame(columns, index=profile.index)


PRICE_LAGS = lear_config["price_lags"]     # [1, 2, 3, 7]
EXO_LAGS = lear_config["exo_lags"]         # [0, 1, 7]; lag 0 = the forecasts FOR day D,
                                           # published before 12:00 of D-1: not leakage
EXOGENOUS_PREFIX = {"load_forecast_mw": "load", "residual_load_mw": "rl"}
assert list(EXOGENOUS_PREFIX) == lear_config["exogenous"]

lag_matrix = pd.concat(
    [lagged_block(daily_profile(dataset, TARGET), PRICE_LAGS, "p")]
    + [lagged_block(daily_profile(dataset, column), EXO_LAGS, prefix)
       for column, prefix in EXOGENOUS_PREFIX.items()],
    axis=1,
)
dataset = dataset.merge(lag_matrix, left_on="local_date", right_index=True,
                        how="left", validate="many_to_one")

LEAR_FEATURES = list(lag_matrix.columns)   # no gas: lear_v2 was rejected in 09
assert not lear_config["gas"]
assert len(LEAR_FEATURES) + 6 == lear_config["n_features"]   # + day-of-week dummies

# --- Checks, this time on a day inside the TEST year --------------------------------
check_day = pd.Timestamp("2025-05-15")
previous_19 = dataset.loc[(dataset["local_date"] == check_day - pd.Timedelta(days=1))
                          & (dataset["local_hour"] == 19), TARGET]
assert np.allclose(dataset.loc[dataset["local_date"] == check_day, "p_d1_h19"].unique(),
                   previous_19.to_numpy()), "p_d1_h19 is not yesterday's 19:00 price"
assert dataset.loc[dataset["split"] != "train", LEAR_FEATURES].notna().all().all()
print(f"lag matrix rebuilt: {len(LEAR_FEATURES)} columns")


def make_lear(features: list, criterion: str = "aic"):
    """24 hourly LASSO regressions, same factory as 09 (Lago et al., 2021).

    All three ingredients are estimated on the training window only:
      1. asinh((price - median) / MAD) on the target and on every lagged price;
      2. standardisation of every column -- a LASSO has ONE penalty for all
         coefficients, so it is not scale-invariant the way OLS is;
      3. LassoLarsIC: the LASSO path by LARS, alpha picked by an information criterion.
    Day-of-week dummies are appended AFTER standardisation and stay 0/1.
    """
    price_index = [i for i, col in enumerate(features) if col.startswith("p_d")]

    def design(df: pd.DataFrame, models: dict | None = None) -> np.ndarray:
        X = df[features].astype(float).to_numpy()
        if models is not None:
            X[:, price_index] = np.arcsinh(
                (X[:, price_index] - models["median"]) / models["mad"])
            X = (X - models["mu"]) / models["sd"]
        dummies = np.stack([(df["day_of_week"].to_numpy() == d).astype(float)
                            for d in range(1, 7)], axis=1)
        return np.hstack([X, dummies])

    def fit(train: pd.DataFrame) -> dict:
        train = train.dropna(subset=features + [TARGET])
        median = train[TARGET].median()
        mad = (train[TARGET] - median).abs().median()
        models = {"median": median, "mad": mad}

        X_raw = train[features].astype(float).to_numpy()
        X_raw[:, price_index] = np.arcsinh((X_raw[:, price_index] - median) / mad)
        models["mu"], models["sd"] = X_raw.mean(axis=0), X_raw.std(axis=0)
        models["sd"][models["sd"] == 0] = 1.0   # a constant column would divide by zero

        X = design(train, models)
        y = np.arcsinh((train[TARGET].to_numpy() - median) / mad)
        hours = train["local_hour"].to_numpy()
        for hour in range(24):
            rows = hours == hour
            models[hour] = LassoLarsIC(criterion=criterion).fit(X[rows], y[rows])
        return models

    def predict(models: dict, day: pd.DataFrame) -> np.ndarray:
        X = design(day, models)
        hours = day["local_hour"].to_numpy()
        forecast = np.empty(len(day))
        for hour in np.unique(hours):
            rows = hours == hour
            z = models[hour].predict(X[rows])
            forecast[rows] = np.sinh(z) * models["mad"] + models["median"]
        return forecast

    return fit, predict, design


# --- lear_v1_avg: refit every 30 days, mean of expanding and rolling 365 -------------
fit_lear, predict_lear, _ = make_lear(LEAR_FEATURES, lear_config["estimator"].split("'")[1])
assert lear_config["averaged_windows"] == [None, ROLLING_WINDOW_DAYS]

dataset["test_lear_expanding"] = walk_forward_test(
    fit_lear, predict_lear, label="lear expanding",
    refit_every_days=lear_config["refit_every_days"],
)
dataset["test_lear_roll365"] = walk_forward_test(
    fit_lear, predict_lear, label="lear rolling 365",
    window_days=ROLLING_WINDOW_DAYS,
    refit_every_days=lear_config["refit_every_days"],
)
dataset["test_lear_v1_avg"] = dataset[
    ["test_lear_expanding", "test_lear_roll365"]].mean(axis=1)

# --- The fourth candidate: one ensemble per XGBoost seed ----------------------------
test_ensemble_columns = []
for seed in SEEDS:
    column = f"test_ensemble_avg_seed{seed}"
    dataset[column] = dataset[["test_linear_v3_avg", f"test_xgb_single_seed{seed}",
                               "test_lear_v1_avg"]].mean(axis=1)
    test_ensemble_columns.append(column)

# Same shape guard the walk-forwards get: present on test, absent everywhere else
is_test = dataset["split"] == "test"
assert dataset.loc[is_test, test_ensemble_columns].notna().all().all()
assert dataset.loc[~is_test, test_ensemble_columns].isna().all().all()

print("\nall four candidates generated on the test year, still not scored")

lag matrix rebuilt: 240 columns
lear expanding: 21 s
lear rolling 365: 20 s

all four candidates generated on the test year, still not scored


In [6]:
# --- The single evaluation on the test year ------------------------------------------
# From here the test set is spent. Nothing below was decided after seeing a number.
TEST_COLUMNS = {
    "linear_v3_avg": ["test_linear_v3_avg"],
    "xgb_single":    [f"test_xgb_single_seed{seed}" for seed in SEEDS],
    "lear_v1_avg":   ["test_lear_v1_avg"],
    "ensemble_avg":  test_ensemble_columns,
}
VALIDATION_COLUMNS = {
    "linear_v3_avg": ["linear_v3_avg"],
    "xgb_single":    xgb_seed_columns,
    "lear_v1_avg":   ["lear_v1_avg"],
    "ensemble_avg":  ensemble_columns,
}
assert list(TEST_COLUMNS) == list(CANDIDATES), "the candidate set changed since cell 1"


def summarise(columns_by_candidate: dict, split: str) -> pd.DataFrame:
    """Mean metrics per candidate, plus the spread across seeds where there is one."""
    flat = [column for columns in columns_by_candidate.values() for column in columns]
    metrics = fc.evaluate(dataset, ["naive_epf"] + flat, split=split)
    overall = metrics.loc[metrics["period"] == "all"].set_index("forecast")

    rows = {}
    for candidate, columns in columns_by_candidate.items():
        block = overall.loc[columns, ["MAE", "RMSE", "WAPE_pct", "rMAE"]]
        row = block.mean().to_dict()
        row["MAE_std"] = block["MAE"].std(ddof=1) if len(columns) > 1 else np.nan
        rows[candidate] = row
    rows["naive_epf"] = overall.loc["naive_epf",
                                    ["MAE", "RMSE", "WAPE_pct", "rMAE"]].to_dict()
    return pd.DataFrame(rows).T


test_summary = summarise(TEST_COLUMNS, "test")
validation_summary = summarise(VALIDATION_COLUMNS, "validation")

print("=== TEST (1 Oct 2024 - 30 Sep 2025) ===")
print(test_summary.round(3).to_string())

# --- How much of the validation result survived? --------------------------------------
# This is the whole point of having kept the test set closed: the validation MAE of a
# model chosen ON the validation year is optimistic by construction. The gap measures
# that optimism plus whatever changed between the two years.
comparison = pd.DataFrame({
    "MAE_validation": validation_summary["MAE"],
    "MAE_test": test_summary["MAE"],
    "rMAE_validation": validation_summary["rMAE"],
    "rMAE_test": test_summary["rMAE"],
})
comparison["MAE_gap"] = comparison["MAE_test"] - comparison["MAE_validation"]
print("\n=== validation vs test ===")
print(comparison.round(3).to_string())
print("\nNote: the naive benchmark also moved between the two years, so rMAE is the "
      "comparable column; MAE alone mixes model decay with a different year.")

# --- Diebold-Mariano: the eight declared comparisons ----------------------------------
# For any comparison involving a seeded model the test is repeated seed by seed, and
# the seed-i ensemble is paired with the seed-i XGBoost it contains. A conclusion that
# holds for some seeds and not others is reported as such, not rounded away.
def dm_summary(candidate_a: str, candidate_b: str, loss: str) -> dict:
    columns_a = TEST_COLUMNS.get(candidate_a, [candidate_a])
    columns_b = TEST_COLUMNS.get(candidate_b, [candidate_b])
    n_runs = max(len(columns_a), len(columns_b))
    runs = [
        fc.diebold_mariano(dataset,
                           columns_a[i % len(columns_a)],
                           columns_b[i % len(columns_b)],
                           split="test", loss=loss)
        for i in range(n_runs)
    ]
    p_values = [run["p_value_b_better"] for run in runs]
    return {
        "A": candidate_a, "B (claimed better)": candidate_b, "loss": loss,
        "runs": n_runs,
        "dm_mean": np.mean([run["dm_stat"] for run in runs]),
        "p_min": min(p_values), "p_max": max(p_values),
        "B wins at 5%": "all" if max(p_values) < 0.05
                        else ("none" if min(p_values) >= 0.05 else "some seeds"),
    }


COMPARISONS = [
    ("naive_epf", "linear_v3_avg"),      # does each candidate beat the benchmark at all
    ("naive_epf", "xgb_single"),
    ("naive_epf", "lear_v1_avg"),
    ("naive_epf", "ensemble_avg"),
    ("linear_v3_avg", "xgb_single"),     # NB: the linear model refits daily, xgb every 30
    ("linear_v3_avg", "lear_v1_avg"),
    ("xgb_single", "lear_v1_avg"),
    ("lear_v1_avg", "ensemble_avg"),     # the question this notebook exists for
]

dm_test = pd.DataFrame([dm_summary(a, b, loss)
                        for a, b in COMPARISONS
                        for loss in ["abs", "squared"]])
print("\n=== Diebold-Mariano on the test year ===")
print(dm_test.round(4).to_string(index=False))
print(f"\nDeclared handicap: linear_v3_avg refits daily, xgb_single and lear_v1_avg "
      f"every 30 days (~{REFIT_HANDICAP_EUR_MWH:.2f} EUR/MWh in favour of the linear model).")

=== TEST (1 Oct 2024 - 30 Sep 2025) ===
                  MAE    RMSE  WAPE_pct   rMAE  MAE_std
linear_v3_avg   9.105  13.020     7.638  0.734      NaN
xgb_single      8.977  12.968     7.530  0.724    0.034
lear_v1_avg     8.280  12.170     6.945  0.667      NaN
ensemble_avg    8.172  11.837     6.855  0.659    0.010
naive_epf      12.405  19.116    10.405  1.000      NaN

=== validation vs test ===
               MAE_validation  MAE_test  rMAE_validation  rMAE_test  MAE_gap
linear_v3_avg           9.317     9.105            0.758      0.734   -0.212
xgb_single              9.127     8.977            0.742      0.724   -0.150
lear_v1_avg             9.030     8.280            0.734      0.667   -0.750
ensemble_avg            8.373     8.172            0.681      0.659   -0.201
naive_epf              12.296    12.405            1.000      1.000    0.108

Note: the naive benchmark also moved between the two years, so rMAE is the comparable column; MAE alone mixes model decay with a diff

In [7]:
# --- Post-test diagnostics 1: what happened to the ensemble? -------------------------
# On validation the ensemble beat the best single model by 0.66 EUR/MWh (p ~ 0); on
# test the gap is 0.11 and not significant on the absolute loss. Two candidate
# explanations, and they are distinguishable: (a) error correlations rose, so there
# was less left to diversify; (b) the components stopped being equally accurate, and
# equal weights pay for that. Nothing here changes a decision: the test is spent.
TEST_SEED0 = ["test_linear_v3_avg", "test_xgb_single_seed0", "test_lear_v1_avg"]
VALIDATION_SEED0 = ["linear_v3_avg", "xgb_single_seed0", "lear_v1_avg"]
LABELS = ["linear", "xgb", "lear"]


def error_frame(columns: list, split: str) -> pd.DataFrame:
    part = dataset.loc[dataset["split"] == split]
    errors = part[columns].apply(lambda column: part[TARGET] - column)
    errors.columns = LABELS
    return errors


errors_by_split = {
    "validation": error_frame(VALIDATION_SEED0, "validation"),
    "test": error_frame(TEST_SEED0, "test"),
}

for split, errors in errors_by_split.items():
    rho = errors.corr()
    rho_bar = rho.to_numpy()[np.triu_indices(3, k=1)].mean()
    sd = errors.std(ddof=0)
    print(f"=== {split} ===")
    print(rho.round(3).to_string())
    print(f"error sd : {sd.round(3).to_dict()}")
    print(f"mean pairwise error correlation : {rho_bar:.3f}")
    print(f"accuracy spread (max sd / min sd): {sd.max() / sd.min():.3f}\n")

# --- Post-hoc ORACLE weights: what equal weights cost on this test year --------------
# Minimum-variance weights computed ON the test set. This is NOT a model and not a
# result: it is an upper bound, an answer to "how much was left on the table". Out of
# sample the weights would have to be estimated, and estimation error usually eats the
# gain (DeMiguel, Garlappi & Uppal 2009, on why 1/N beats estimated mean-variance).
E = errors_by_split["test"].to_numpy()
second_moments = E.T @ E / len(E)
weights = np.linalg.solve(second_moments, np.ones(3))
weights /= weights.sum()
oracle_error = E @ weights

print("ORACLE minimum-variance weights on the test year (post-hoc, not a model):")
print(pd.Series(weights, index=LABELS).round(3).to_string())
print(f"\n         equal weights | oracle | best single")
print(f"RMSE   {np.sqrt((E.mean(axis=1) ** 2).mean()):14.3f} | "
      f"{np.sqrt((oracle_error ** 2).mean()):6.3f} | "
      f"{errors_by_split['test'].pow(2).mean().min() ** 0.5:11.3f}")
print(f"MAE    {np.abs(E.mean(axis=1)).mean():14.3f} | "
      f"{np.abs(oracle_error).mean():6.3f} | "
      f"{errors_by_split['test'].abs().mean().min():11.3f}")

=== validation ===
        linear    xgb   lear
linear   1.000  0.879  0.673
xgb      0.879  1.000  0.692
lear     0.673  0.692  1.000
error sd : {'linear': 13.083, 'xgb': 12.94, 'lear': 12.89}
mean pairwise error correlation : 0.748
accuracy spread (max sd / min sd): 1.015

=== test ===
        linear    xgb   lear
linear   1.000  0.865  0.799
xgb      0.865  1.000  0.733
lear     0.799  0.733  1.000
error sd : {'linear': 12.995, 'xgb': 12.932, 'lear': 12.168}
mean pairwise error correlation : 0.799
accuracy spread (max sd / min sd): 1.068

ORACLE minimum-variance weights on the test year (post-hoc, not a model):
linear    0.052
xgb       0.353
lear      0.594

         equal weights | oracle | best single
RMSE           11.831 | 11.635 |      12.170
MAE             8.166 |  7.979 |       8.280


In [8]:
# --- Post-test diagnostics 2: why did EVERY model improve from validation to test? ----
# Every candidate has a negative MAE gap while the naive benchmark got worse. Two
# mechanisms are mixed. A rolling-365 model trains on exactly 365 days in BOTH years,
# so its change measures only "this year was easier to forecast". The expanding version
# additionally receives a whole extra year of history, so the DIFFERENCE between the
# two changes is what the extra data was worth.
EXTRA_VALIDATION = {
    "linear_validation_forecasts.parquet": ["linear_v3", "linear_v3_roll365"],
    "lear_validation_forecasts.parquet": ["lear_v1", "lear_v1_roll365"],
}
for filename, columns in EXTRA_VALIDATION.items():
    saved = pd.read_parquet(fc.PROCESSED / filename)
    dataset = dataset.merge(saved[["ts_utc"] + columns], on="ts_utc",
                            how="left", validate="one_to_one")

# Each pair is the SAME recipe and the SAME protocol on the two years
WINDOW_PAIRS = {
    "naive_epf":          ("naive_epf", "naive_epf"),
    "linear expanding":   ("linear_v3", "test_linear_expanding"),
    "linear rolling 365": ("linear_v3_roll365", "test_linear_roll365"),
    "lear expanding":     ("lear_v1", "test_lear_expanding"),
    "lear rolling 365":   ("lear_v1_roll365", "test_lear_roll365"),
}

def mae(column: str, split: str) -> float:
    part = dataset.loc[dataset["split"] == split]
    return (part[TARGET] - part[column]).abs().mean()

decomposition = pd.DataFrame([
    {"model": label,
     "MAE_validation": mae(validation_column, "validation"),
     "MAE_test": mae(test_column, "test")}
    for label, (validation_column, test_column) in WINDOW_PAIRS.items()
])
decomposition["gap"] = decomposition["MAE_test"] - decomposition["MAE_validation"]
print(decomposition.round(3).to_string(index=False))
print("\nrolling gap  = effect of the year alone (same 365 training days both times)")
print("expanding gap = year + one extra year of training history")
print("difference    = what the extra data bought that model")

             model  MAE_validation  MAE_test    gap
         naive_epf          12.296    12.405  0.108
  linear expanding           9.660     9.507 -0.154
linear rolling 365           9.511     9.158 -0.354
    lear expanding           9.296     8.322 -0.974
  lear rolling 365           9.586     9.087 -0.499

rolling gap  = effect of the year alone (same 365 training days both times)
expanding gap = year + one extra year of training history
difference    = what the extra data bought that model


In [9]:
# --- Persist the test results and the protocol that produced them ---------------------
import json

is_test = dataset["split"] == "test"
KEEP = ["ts_utc", "ts_local", "local_date", "local_hour", "period", "split", TARGET]
RESULT_COLUMNS = (
    ["test_linear_expanding", "test_linear_roll365", "test_linear_v3_avg",
     "test_lear_expanding", "test_lear_roll365", "test_lear_v1_avg"]
    + [f"test_xgb_single_seed{seed}" for seed in SEEDS]
    + test_ensemble_columns
)

forecasts_out = dataset.loc[is_test, KEEP + NAIVE_COLUMNS + RESULT_COLUMNS].reset_index(drop=True)
forecasts_path = fc.PROCESSED / "final_test_forecasts.parquet"
forecasts_out.to_parquet(forecasts_path, index=False)
pd.testing.assert_frame_equal(pd.read_parquet(forecasts_path), forecasts_out)

fc.evaluate(dataset, NAIVE_COLUMNS + RESULT_COLUMNS, split="test").to_csv(
    fc.RESULTS / "10_final_test_metrics.csv", index=False)
dm_test.to_csv(fc.RESULTS / "10_final_test_dm.csv", index=False)
test_summary.to_csv(fc.RESULTS / "10_final_test_summary.csv")

# The record of what was decided BEFORE the test was opened, stored NEXT TO the results:
# without it, "we declared the ensemble in advance" is just a claim.
protocol = {
    "declared_on": "2026-09-18",
    "candidates": CANDIDATES,
    "comparisons": [list(pair) for pair in COMPARISONS],
    "losses": ["abs", "squared"],
    "seeds": SEEDS,
    "refit_handicap_eur_mwh": REFIT_HANDICAP_EUR_MWH,
    "test_period": ["2024-10-01", "2025-09-30"],
    "findings": {
        "lowest_mae": "ensemble_avg",
        "ensemble_beats_best_single": False,     # p 0.14-0.20 (abs)
        "xgb_beats_linear": False,               # p 0.09-0.26 (abs)
        "lear_beats_xgb": True,                  # p <= 0.0003 (abs)
        "rank_changed_from_validation": True,
    },
}
(fc.RESULTS / "10_final_test_protocol.json").write_text(json.dumps(protocol, indent=2))

print(f"saved {forecasts_out.shape} -> {forecasts_path.name}")
for name in ["metrics.csv", "dm.csv", "summary.csv", "protocol.json"]:
    print(f"saved -> results/10_final_test_{name}")

saved (8760, 27) -> final_test_forecasts.parquet
saved -> results/10_final_test_metrics.csv
saved -> results/10_final_test_dm.csv
saved -> results/10_final_test_summary.csv
saved -> results/10_final_test_protocol.json


## Summary — the test year, scored once

Four candidates were declared before the test set was opened. On the test year the
ensemble has the lowest MAE (8.17, rMAE 0.659) but its edge over the best single model
is not significant; XGBoost does not detectably beat an hourly linear regression; LEAR
beats both with p ≈ 0. The ranking chosen on validation is not the one the test
confirms, and the cause is measured rather than assumed: LEAR is the only model
parameterised heavily enough to profit from the extra year of history, while the linear
model is hurt by it.

### What was fixed in advance

| Decided | What |
|---|---|
| 07 / 08 / 09 | `linear_v3_avg`, `xgb_single`, `lear_v1_avg`, each chosen on validation |
| 18 Sep, before any test forecast | `ensemble_avg` = equal-weighted mean of the three; no weight estimated |
| 18 Sep | 8 comparisons, 2 loss functions, 5 XGBoost seeds, one ensemble per seed |
| 18 Sep | Refit protocols declared, not harmonised: the linear model refits daily, the other two every 30 days (~0.10 EUR/MWh in the linear model's favour) |

Written to `results/10_final_test_protocol.json` alongside the results, so the claim
"this was declared in advance" is auditable rather than asserted.

### Results (1 Oct 2024 – 30 Sep 2025, 8,760 hours)

| Candidate | MAE | RMSE | WAPE % | rMAE |
|---|---|---|---|---|
| `naive_epf` | 12.405 | 19.116 | 10.41 | 1.000 |
| `linear_v3_avg` | 9.105 | 13.020 | 7.64 | 0.734 |
| `xgb_single` | 8.977 ± 0.034 | 12.968 | 7.53 | 0.724 |
| `lear_v1_avg` | 8.280 | 12.170 | 6.95 | 0.667 |
| **`ensemble_avg`** | **8.172 ± 0.010** | **11.837** | **6.86** | **0.659** |

Diebold-Mariano, daily loss differentials, one-sided:

| A | B | abs | squared | B wins at 5% |
|---|---|---|---|---|
| `naive_epf` | all four | p ≈ 0 | p ≈ 0 | all |
| `linear_v3_avg` | `xgb_single` | p 0.094–0.262 | p 0.346–0.506 | **none** |
| `linear_v3_avg` | `lear_v1_avg` | p ≈ 0 | p 0.0006 | all |
| `xgb_single` | `lear_v1_avg` | p 0.0001–0.0003 | p 0.010–0.022 | all |
| `lear_v1_avg` | `ensemble_avg` | p 0.143–0.196 | p 0.042–0.055 | **some seeds** |

XGBoost carries a protocol handicap that runs *against* it, so correcting for it would
widen its 0.128 EUR/MWh lead over the linear model to roughly 0.23 — still not
significant. The conclusion is therefore conservative, not convenient.

### Selection optimism showed up as rank instability, not as decay

| | MAE validation | MAE test | gap | rMAE validation | rMAE test |
|---|---|---|---|---|---|
| `linear_v3_avg` | 9.317 | 9.105 | −0.212 | 0.758 | 0.734 |
| `xgb_single` | 9.127 | 8.977 | −0.150 | 0.742 | 0.724 |
| `lear_v1_avg` | 9.030 | 8.280 | **−0.750** | 0.734 | 0.667 |
| `ensemble_avg` | 8.373 | 8.172 | −0.201 | 0.681 | 0.659 |
| `naive_epf` | 12.296 | 12.405 | +0.108 | 1.000 | 1.000 |

Every model improved while the naive benchmark got worse: the test year was not easier
in general, it rewarded structural forecasting over day-on-day persistence. The
validation ordering (ensemble ≫ lear ≈ xgb ≈ linear) became lear ≈ ensemble ≫ xgb ≈
linear. Holding the test set closed did not reveal an inflated level — it revealed an
unstable ranking, which is the subtler failure mode.

### Why LEAR jumped: parameter count, not luck

A rolling-365 model trains on exactly 365 days in both years, so its change measures the
year alone. The expanding version also receives one extra year of history, so the
difference between the two changes is what the data was worth.

| Model | MAE validation | MAE test | gap | value of the extra year |
|---|---|---|---|---|
| `naive_epf` | 12.296 | 12.405 | +0.108 | — |
| linear expanding | 9.660 | 9.507 | −0.154 | |
| linear rolling 365 | 9.511 | 9.158 | −0.354 | **+0.200 (worse)** |
| LEAR expanding | 9.296 | 8.322 | −0.974 | |
| LEAR rolling 365 | 9.586 | 9.087 | −0.499 | **−0.475 (better)** |

Same extra data, opposite sign. LEAR fits 246 coefficients per hour on one row per day:
at 365 days it is nearly undetermined, so every extra observation cuts estimation
variance. The hourly linear model fits ~12 coefficients on ~365 rows per hour and is
already well determined, so extra history adds no variance reduction and only bias — the
2021/22 gas crisis, a regime that is not coming back. Textbook bias–variance, measured
on two models trained on the same data.

### Why the ensemble lost most of its edge

Equal-weight combination needs components of comparable accuracy, and both requirements
moved against it between the two years:

| | validation | test |
|---|---|---|
| mean pairwise error correlation | 0.748 | 0.799 |
| accuracy spread (max sd / min sd) | 1.015 | 1.068 |
| gain vs the **average** component | −8.8% | −6.9% |
| gain vs the **best** component | −8.2% | −2.8% |

Rising correlation (driven by linear–LEAR, 0.673 → 0.799) cost 1.9 points; the widening
accuracy spread cost another 4.1. The imbalance did more than twice the damage of the
correlation.

Post-hoc oracle check (minimum-variance weights computed *on* the test set — an upper
bound, not a model): weights LEAR 0.594, XGBoost 0.353, linear 0.052, i.e. the linear
model is dropped as redundant with XGBoost (error correlation 0.865). MAE best single
8.280 → equal weights 8.166 → oracle 7.979. Combination was worth 0.301 EUR/MWh in
total and equal weights captured 38% of it. The remaining 0.187 would have to be won
out of sample while paying for weight estimation — see DeMiguel, Garlappi & Uppal
(2009) on 1/N versus estimated mean-variance. Equal weights were a defensible choice,
not a lazy one.

The same rule appears one level down: inside LEAR, averaging the two calibration windows
is worth 0.266 on validation (components 0.290 apart) and only 0.042 on test (components
0.765 apart).

### Limits to state with the results

- The target is the **PUN**, a national index; the tradeable price is **zonal**.
- One test year (366 → 365 days) limits Diebold-Mariano power; a borderline p-value here
  is genuinely borderline.
- Both validation and test sit in a calm gas regime. The `09` finding that explicit gas
  adds nothing to LEAR was already caveated for low power; this test does not lift that
  caveat.
- The ensemble's advantage is a point estimate, not a demonstrated one.